In [2]:
import os
import sys
import pandas as pd

# This allows our notebook to look outside the notebooks/ folder and import from src/
sys.path.append(os.path.abspath("../src"))
from ingest import DataIngestionEngine

# 1. Pipeline execution using relative paths
base_dir = os.path.abspath("..")
data_path = os.path.join(base_dir, "data", "raw_transactions.csv")
engine = DataIngestionEngine(data_path)
clean_data = engine.process_pipeline()

# 2. Load into a Pandas DataFrame
df = pd.DataFrame(clean_data)
print("\nDataFrame Loaded Successfully!")
df.head()

🚀 Starting Data Ingestion Pipeline...
✅ Pipeline Completed! Successfully ingested 4 clean records.

DataFrame Loaded Successfully!


,transaction_id,customer_id,timestamp,amount,product_category,status
0,T1001,C101,2026-06-01 10:15:30,250.50,Electronics,COMPLETED
1,T1004,C104,2026-06-02 09:00:15,120.00,Apparel,PENDING
2,T1006,C102,2026-06-02 16:35:40,500.00,Electronics,COMPLETED
3,T1007,C105,2026-06-03 18:22:11,15.25,Books,COMPLETED


In [4]:
# Group by product category and calculate total revenue, average order value, and volume
category_matrix = df.groupby("product_category")["amount"].agg(["sum", "mean", "count"]).rename(
    columns={"sum": "Total Revenue", "mean": "Average Order Value", "count": "Order Volume"}
)

# Display a beautifully styled table inside the notebook using gradients
category_matrix.style.background_gradient(cmap="Blues", subset=["Total Revenue"])

,Total Revenue,Average Order Value,Order Volume
product_category,,,
Apparel,120.000000,120.000000,1
Books,15.250000,15.250000,1
Electronics,750.500000,375.250000,2


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
import numpy as np

print("🤖 Initializing Machine Learning Workflow...")

# 1. Feature Selection (X = Inputs, y = Target Output we want to predict)
# We want to use 'product_category' to predict the transaction 'amount'
X = df[['product_category']]
y = df['amount']

# 2. Data Transformation (One-Hot Encoding)
# Machine learning models only understand numbers, not text like "Apparel" or "Electronics".
# OneHotEncoder converts text categories into distinct binary columns (0s and 1s).
encoder = OneHotEncoder(sparse_output=False)
X_encoded = encoder.fit_transform(X)

# 3. Initialize and Train our Linear Regression AI Model
model = LinearRegression()
model.fit(X_encoded, y)
print("✅ AI Model Training Complete!")

# 4. Run a Live Prediction Simulation
# Let's see what the model predicts a customer will spend if they walk in to buy Electronics!
test_category = np.array([['Electronics']])
test_encoded = encoder.transform(test_category)
predicted_amount = model.predict(test_encoded)

print(f"\n🔮 AI Prediction Result:")
print(f"For a customer browsing 'Electronics', the model predicts an expenditure of: ${predicted_amount[0]:.2f}")

🤖 Initializing Machine Learning Workflow...
✅ AI Model Training Complete!

🔮 AI Prediction Result:
For a customer browsing 'Electronics', the model predicts an expenditure of: $375.25


c:\Projects\smart-analytics-predictive-engine\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


In [7]:
from sklearn.metrics import mean_squared_error, r2_score

# 1. Generate predictions for all our historical data rows
y_pred = model.predict(X_encoded)

# 2. Calculate our performance scores
mse = mean_squared_error(y, y_pred)
rmse = np.sqrt(mse) # Root Mean Squared Error brings the unit back to normal dollars
r2 = r2_score(y, y_pred)

print("📊 MODEL PERFORMANCE METRICS 📊")
print("---------------------------------")
print(f"Mean Squared Error (MSE):       {mse:.4f}")
print(f"Root Mean Squared Error (RMSE): ${rmse:.2f}")
print(f"R-squared (R2 Score):           {r2:.4f} ({r2*100:.1f}%)")

📊 MODEL PERFORMANCE METRICS 📊
---------------------------------
Mean Squared Error (MSE):       7781.2812
Root Mean Squared Error (RMSE): $88.21
R-squared (R2 Score):           0.7628 (76.3%)
